In [3]:
#all imports necessary for the project
import numpy as np
import cv2
import os
import tensorflow as tf

from tensorflow.keras import layers, models, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
#configuring image size and batch size for tthe model, compatible with MobileNetV2
IMAGE_SIZE = (224, 224)
IMAGE_FULL_SIZE = (224, 224, 3)
BATCH_SIZE = 8 # the batch size is small due to technical limitations

In [6]:
#dataset paths to the downloaded image folders containing the training and testing images
trainImgfolder = "C:/Users/marto/OneDrive - Budapesti Műszaki és Gazdaságtudományi Egyetem/ML/priject/pp/dogImages/train"
testImgfolder = "C:/Users/marto/OneDrive - Budapesti Műszaki és Gazdaságtudományi Egyetem/ML/priject/pp/dogImages/test"
validImgfolder = "C:/Users/marto/OneDrive - Budapesti Műszaki és Gazdaságtudományi Egyetem/ML/priject/pp/dogImages/valid"

#tensorflow datast loader
def load_image_dataset(
    trainImgfolder,
    img_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    categorical=True,
    shuffle=True
):
    label_mode = "categorical" if categorical else "int" # formatting thelabels to one-hot (integer) labels
#loading the dataset from the folders and formattign the images to the desired size, and batch size, also shuffling the imaes for better ttraining
    ds = tf.keras.utils.image_dataset_from_directory(
        trainImgfolder,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode=label_mode,
        shuffle=shuffle
    )


#extracting thhe class names(dog breeds)
    class_names = ds.class_names
    return ds, class_names


In [7]:
#loads the test dataset as tensorflow dataset with the same format as the training dataset
test_ds, class_names = load_image_dataset(
    testImgfolder,
    categorical=True,
    shuffle=True
)

Found 836 files belonging to 133 classes.


In [8]:
#manual dataset loader used for training
def load_dataset(root_dir, img_size=(224, 224)):
    X = [] #images
    y = []  #labels
    class_names = []
#iterating over each class folder
    for class_name in sorted(os.listdir(root_dir)):
        class_path = os.path.join(root_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        class_names.append(class_name)
        label = len(class_names) - 1 #assigning numerical label

#loading all images from the lass folder
        for fname in os.listdir(class_path):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            path = os.path.join(class_path, fname)
#reading image using OpenCV
            img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)

            if img is None:
                continue
#converting BGR-> RGB as openCV uses BGR by default, resize to model input size
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)

            X.append(img)
            y.append(label)
#normalizepixel values to [0,1] for stable NN training, convert to float16 to save memory, and convert labels to int32
    X = np.array(X, dtype=np.float16) / 255.0
    y = np.array(y, dtype=np.int32)

    return X, y, class_names 

In [9]:
#loading datasets into memory
X_train, y_train, class_names = load_dataset(trainImgfolder)
X_test, y_test, _ = load_dataset(testImgfolder)
X_val, y_val, _ = load_dataset(validImgfolder)

#checking dataset shapes to make sure they are loaded correctly
print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

(6679, 224, 224, 3)
(836, 224, 224, 3)
(835, 224, 224, 3)


In [7]:
#number of classes (dog breeds)
numOfCategories = len(np.unique(y_train))
print(numOfCategories)

133


In [8]:
#MoobilNetV2 pretrained model used to exttract features (transfer learnign)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_FULL_SIZE,
    include_top=False, #remoe classification head
    weights='imagenet' #use pretrained weights
)
#freezing convolutional base to prevent overfitting
base_model.trainable = False
#custom clssificationhead 
model = models.Sequential([
    base_model,
    #convert feature maps
    layers.GlobalAveragePooling2D(),
    #regularization toreduce overfitting
    layers.Dropout(0.4),
    #final classiffier output layer with softmax activation
    layers.Dense(numOfCategories, activation='softmax')
])

#compiling the model using ADAM optimizer with a low learning rate
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 133)            │       170,373 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,428,357 (9.26 MB)

 Trainable params: 170,373 (665.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
numOfCategories = len(np.unique(y_train))

#convertin integer labels as categorical format is required for sftmax training
y_train_cat = to_categorical(y_train, num_classes=numOfCategories)
y_test_cat  = to_categorical(y_test, num_classes=numOfCategories)
y_val_cat  = to_categorical(y_val, num_classes=numOfCategories)

# compile model
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

In [ ]:
#saving the best models weight for testing later and callbacks for early stopping and learning rate reduction
best_model_file = "C:/temp/dogs/best_model.h5"

callbacks = [
    #save when model validation improves
    ModelCheckpoint(best_model_file, save_best_only=True, verbose=1),
    #raducing learning rate on plateau to avoid overshooting the minimum
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.1,
        patience=3,
        verbose=1,
        min_lr=1e-7
    ),
    #early stopping to prevent overffitting
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        verbose=1,
        restore_best_weights=True
    )
]

#model training
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=40,
    batch_size=BATCH_SIZE,
    callbacks=callbacks  
)

In [ ]:
#loading the best model
model = load_model("C:/temp/dogs/best_model.h5")

#recompiling
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

#evaluate accuracy on the test set
loss, accuracy = model.evaluate(X_test, y_test_cat)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

#predict class probabilities for test images
predictions = model.predict(X_test)